Install the libraries

In [ ]:
!pip install scikit-learn
!pip install torch

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# ---------------- CONFIG ----------------
INPUT_FILE = "datasets/clean_eye_data.npz"
SEQ_LEN = 3   # number of windows per sequence

# ---------------- LOAD CLEAN DATA ----------------
data = np.load(INPUT_FILE)

X_clean = data["X"]   # (N, 8, 250)
x_feat = data["X_feat"]    # (N, num_features)
y_clean = data["y"]   # (N,)

y_clean = y_clean - 1

print("Loaded clean data:")
print("X:", X_clean.shape)
print("y:", y_clean.shape)
print("Label distribution:", np.unique(y_clean, return_counts=True))

X_train, X_val, y_train, y_val, feat_train, feat_val = train_test_split(
    X_clean,
    y_clean,
    x_feat,
    test_size=0.3,
    stratify=y_clean,
    random_state=42
)

print("After channel selection:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("feat_train:", feat_train.shape)
print("feat_val:", feat_val.shape)


In [ ]:
def build_lstm_sequences(X, y, seq_len):
    N, C, T = X.shape

    X_seq = []
    y_seq = []

    for i in range(N - seq_len + 1):
        seq = X[i:i + seq_len]              # (seq_len, C, T)
        seq_flat = seq.reshape(seq_len, -1) # (seq_len, C*T)

        X_seq.append(seq_flat)
        y_seq.append(y[i + seq_len - 1])    # label of last window

    return np.array(X_seq), np.array(y_seq)

def build_feature_sequences(X_feat, y, seq_len):
    X_seq = []
    y_seq = []

    for i in range(len(X_feat) - seq_len + 1):
        X_seq.append(X_feat[i:i + seq_len])
        y_seq.append(y[i + seq_len - 1])

    return np.array(X_seq), np.array(y_seq)


# ---------------- BUILD SEQUENCES ----------------
feat_train_seq, y_train_seq_feat = build_feature_sequences(feat_train, y_train, SEQ_LEN)
feat_val_seq,   y_val_seq_feat   = build_feature_sequences(feat_val,   y_val,   SEQ_LEN)

feat_mean = feat_train_seq.mean(axis=(0, 1), keepdims=True)
feat_std  = feat_train_seq.std(axis=(0, 1), keepdims=True) + 1e-6
feat_train_seq = (feat_train_seq - feat_mean) / feat_std
feat_val_seq   = (feat_val_seq   - feat_mean) / feat_std

X_train_seq, y_train_seq = build_lstm_sequences(X_train, y_train, SEQ_LEN)
X_val_seq,   y_val_seq   = build_lstm_sequences(X_val,   y_val,   SEQ_LEN)

raw_mean = X_train_seq.mean(axis=(0,1), keepdims=True)
raw_std  = X_train_seq.std(axis=(0,1), keepdims=True) + 1e-6
RAW_DIM = 500

print("\nLSTM-ready data:")
print("Train:", X_train_seq.shape, y_train_seq.shape)
print("Val:  ", X_val_seq.shape,   y_val_seq.shape)

print("Train label dist:", np.unique(y_train_seq, return_counts=True))
print("Val label dist:  ", np.unique(y_val_seq,   return_counts=True))
print("Raw mean shape:", raw_mean.shape)
print("Raw std shape:", raw_std.shape)

Training The Neural Network, Set Up Data

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class EEGHybridDataset(Dataset):
    def __init__(self, X_raw, X_feat, y):
        self.X_raw  = torch.tensor(X_raw,  dtype=torch.float32)
        self.X_feat = torch.tensor(X_feat, dtype=torch.float32)
        self.y      = torch.tensor(y,      dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_feat[idx], self.X_raw[idx], self.y[idx]
    
train_ds = EEGHybridDataset(X_train_seq, feat_train_seq, y_train_seq)
val_ds   = EEGHybridDataset(X_val_seq,   feat_val_seq,   y_val_seq)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

Model Architecture

In [ ]:
import torch
import torch.nn as nn

class EyeClassifier(nn.Module):
    def __init__(
        self,
        feat_dim,        # number of handcrafted features (e.g. 12)
        raw_dim,         # raw EEG per timestep (e.g. C*T or reduced)
        lstm_hidden=64
    ):
        super().__init__()

        # -------- Feature branch (band powers etc.) --------
        self.feature_mlp = nn.Sequential(
            nn.Linear(feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )

        # -------- Raw EEG branch (temporal dynamics) --------
        self.lstm = nn.LSTM(
            input_size=raw_dim,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        # -------- Fusion + classifier --------
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden + 16, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x_feat, x_raw):
        """
        x_feat: (B, T, feat_dim)   or (B, feat_dim) if already pooled
        x_raw:  (B, T, raw_dim)
        """

        # ---- Feature branch ----
        if x_feat.dim() == 3:
            feat_emb = self.feature_mlp(x_feat)   # (B, T, 32)
            feat_last = feat_emb[:, -1, :]        # (B, 32)
        else:
            feat_last = self.feature_mlp(x_feat)  # (B, 32)

        # ---- Raw EEG branch ----
        lstm_out, _ = self.lstm(x_raw)             # (B, T, H)
        lstm_last = lstm_out[:, -1, :]             # (B, H)

        # ---- Fusion ----
        fused = torch.cat([feat_last, lstm_last], dim=1)
        return self.classifier(fused)

Training Loop Log Loss every epoch for train and test set

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = EyeClassifier(
    feat_dim=feat_train_seq.shape[2], 
    raw_dim=X_train_seq.shape[2],        
    lstm_hidden=32
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
best_ratio = -float("inf")

In [ ]:
# =====================
# METRIC TRACKING
# =====================
num_epochs = 100
train_losses = []
val_losses = []
train_accs = []
val_accs = []
ratio_history = []

best_ratio_epoch = 0

# =====================
# TRAINING LOOP
# =====================
for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_feat, X_raw, y in train_loader:
        X_feat = X_feat.to(device)
        X_raw  = X_raw.to(device)
        y      = y.to(device)

        optimizer.zero_grad()
        outputs = model(X_feat, X_raw)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * y.size(0)
        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # -------- VALIDATION --------
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X_feat, X_raw, y in val_loader:
            X_feat = X_feat.to(device)
            X_raw  = X_raw.to(device)
            y      = y.to(device)

            outputs = model(X_feat, X_raw)
            loss = criterion(outputs, y)

            val_loss += loss.item() * y.size(0)
            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    # -------- RATIO METRIC --------
    ratio = val_acc / (val_loss + 1e-8)
    ratio_history.append(ratio)

    # -------- SAVE BEST RATIO MODEL --------
    if ratio > best_ratio:
        best_ratio = ratio
        best_ratio_epoch = epoch + 1

        torch.save(
           model, "assets/eye_model.pth"
        )

        print(
            f" 🏆 RATIO BEST saved | "
            f"Acc: {val_acc*100:.2f}% | "
            f"Loss: {val_loss:.4f} | "
            f"Ratio: {ratio:.3f} | "
            f"Epoch {epoch+1}"
        )

    # -------- LOG --------
    print(
        f"Epoch {epoch+1:03d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}% | "
        f"Ratio: {ratio:.3f}"
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

# =====================
# SUMMARY
# =====================
print(f"\nBest accuracy/loss ratio: {best_ratio:.3f}")
print(f"Best ratio epoch: {best_ratio_epoch}")

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ratio_history, label="Accuracy / Loss Ratio")
plt.axvline(best_ratio_epoch - 1, color="r", linestyle="--",
            label=f"Best Epoch {best_ratio_epoch}")
plt.xlabel("Epoch")
plt.ylabel("Acc / Loss Ratio")
plt.title("Ratio over Training")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, [a * 100 for a in train_accs], label="Train Acc")
plt.plot(epochs, [a * 100 for a in val_accs], label="Val Acc")

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Train vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

<h1 align="Center"> Real Time Inference </h1>

In [ ]:
import numpy as np

def noise_augmentation(X):
    N, C, T = X.shape # n windows, n channels, n timepoints

    rms = np.zeros((N, C))
    peak = np.zeros((N, C))
    spike = np.zeros(N)
    corr_mean = np.zeros(N)

    for i in range(N):
        w = X[i]

        rms[i] = np.sqrt(np.mean(w**2, axis=1))
        peak[i] = np.max(np.abs(w), axis=1)

        diff = np.diff(w, axis=1)
        spike[i] = np.max(np.abs(diff))

        corr = np.corrcoef(w)
        upper = corr[np.triu_indices_from(corr, k=1)]
        corr_mean[i] = np.mean(np.abs(upper))

    rms_mu, rms_std = rms.mean(), rms.std()
    peak_mu, peak_std = peak.mean(), peak.std()
    spike_mu, spike_std = spike.mean(), spike.std()

    bad_rms   = np.any(rms > rms_mu + 6 * rms_std, axis=1)
    bad_peak  = np.any(peak > peak_mu + 10 * peak_std, axis=1)
    bad_spike = spike > spike_mu + 10 * spike_std
    bad_corr  = corr_mean > 0.95

    bad = bad_rms | bad_peak | bad_spike | bad_corr
    good = ~bad

    X_clean = X[good]

    X_clean = np.clip(X_clean, -300, 300)  # µV

    return X_clean

In [ ]:
import numpy as np
from scipy.signal import welch

def extract_band_features_per_channel(X, fs=250):
    bands = {"delta": (0.5, 4),"theta": (4, 8),"alpha": (8, 13),"beta":  (13, 30),"gamma": (30, 45),}
    N, C, T = X.shape
    features = []
    feature_names = []

    # ---------- FEATURE NAMES ----------
    for ch in range(C):
        prefix = f"ch{ch}"

        # spectral powers
        for band in bands:
            feature_names.append(f"{prefix}_{band}_power")
        for band in bands:
            feature_names.append(f"{prefix}_log_{band}_power")

        # ratios + complexity
        feature_names += [f"{prefix}_alpha_relative",f"{prefix}_alpha_beta_ratio",f"{prefix}_theta_alpha_ratio",f"{prefix}_beta_alpha_ratio",f"{prefix}_spectral_entropy",f"{prefix}_log_total_power"]

        # time-domain stats (NEW)
        feature_names += [f"{prefix}_time_mean",f"{prefix}_time_std",f"{prefix}_time_var",]

    # ---------- FEATURE EXTRACTION ----------
    for i in range(N):
        feat_vec = []

        for ch in range(C):
            signal = X[i, ch]

            # ----- PSD -----
            f, Pxx = welch(signal, fs=fs, nperseg=fs // 2)

            band_powers = {}
            total_power = 0.0

            for band, (fmin, fmax) in bands.items():
                mask = (f >= fmin) & (f <= fmax)
                power = np.trapezoid(Pxx[mask], f[mask])
                band_powers[band] = power
                total_power += power

            # ----- raw band powers -----
            for band in bands:
                feat_vec.append(band_powers[band])

            # ----- log band powers -----
            for band in bands:
                feat_vec.append(np.log(band_powers[band] + 1e-8))

            # ----- ratios -----
            alpha_rel   = band_powers["alpha"] / (total_power + 1e-8)
            alpha_beta  = band_powers["alpha"] / (band_powers["beta"] + 1e-8)
            theta_alpha = band_powers["theta"] / (band_powers["alpha"] + 1e-8)
            beta_alpha  = band_powers["beta"]  / (band_powers["alpha"] + 1e-8)

            # ----- spectral entropy -----
            Pxx_norm = Pxx / (Pxx.sum() + 1e-8)
            spec_entropy = -np.sum(Pxx_norm * np.log(Pxx_norm + 1e-8))

            log_total_power = np.log(total_power + 1e-8)

            feat_vec += [
                alpha_rel,
                alpha_beta,
                theta_alpha,
                beta_alpha,
                spec_entropy,
                log_total_power,
            ]

            # ----- time-domain stats (NEW) -----
            feat_vec += [
                signal.mean(),
                signal.std(),
                signal.var(),
            ]

        features.append(feat_vec)

    return np.array(features), feature_names

In [ ]:
def build_lstm_sequences_no_y(X, seq_len):
    N, C, T = X.shape

    X_seq = []

    for i in range(N - seq_len + 1):
        seq = X[i:i + seq_len]              # (seq_len, C, T)
        seq_flat = seq.reshape(seq_len, -1) # (seq_len, C*T)
        X_seq.append(seq_flat)

    return np.array(X_seq)

In [59]:
import cv2
from brainflow import DataFilter, FilterTypes, AggOperations
from brainflow.board_shim import BoardShim, BrainFlowInputParams
import time
import numpy as np

BOARD_ID = 0
PORT = "COM3"
FS = 250
params = BrainFlowInputParams()
params.serial_port = PORT

BoardShim.enable_dev_board_logger()
board = BoardShim(BOARD_ID, params)
board.prepare_session()
board.start_stream()

print("Noise Rejection Under Process...")
time.sleep(2)


cam = cv2.VideoCapture(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = np.load("assets/model_scaler.npz")
feat_mean = scaler["feat_mean"]
feat_std = scaler["feat_std"]

old_state = ""

model = torch.load("assets/eye_model.pth", weights_only=False)

while True:

    #Brainflow Stuff
    data = board.get_current_board_data(750)

    if data.shape[1] < 750:
        continue

    eeg_data = data[board.get_eeg_channels(BOARD_ID)]

    #Filtering Data
    test_X = []
    for ch in range(eeg_data.shape[0]):
        DataFilter.perform_bandpass(eeg_data[ch], FS, 0.5, 40.0, 4, FilterTypes.BUTTERWORTH.value, 0)
        DataFilter.perform_bandstop(eeg_data[ch], FS, 48.0, 52.0, 4, FilterTypes.BUTTERWORTH.value, 0)
        for i in range(0, eeg_data.shape[1] - 250, 250):
            test_X.append(eeg_data[:, i:i + 250])

    test_X = np.array(test_X)  # (N, 8, 250)
    # Noise Augmentation

    clean_X = noise_augmentation(test_X)
    if clean_X.shape[0] >= 3:
        pass
    else:
        continue

    # Feature Extraction
    features, _ = extract_band_features_per_channel(clean_X, fs=FS)
    #Build sequences
    features_seq = features[np.newaxis, :, :]  # (1, 3, num_features)
    features_seq = (features_seq - feat_mean) / feat_std

    X_seq = build_lstm_sequences_no_y(clean_X, seq_len=3)[-1:] 
        
    # Pytorch Inference
    with torch.no_grad():
        X_feat_tensor = torch.tensor(features_seq, dtype=torch.float32).to(device)
        X_raw_tensor  = torch.tensor(X_seq, dtype=torch.float32).to(device)

        outputs = model(X_feat_tensor, X_raw_tensor)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        logits = outputs.cpu().numpy()

    #Show Results on web camera feed
    output_text = "Eyes Open" if preds[-1] == 0 else "Eyes Closed"

    if output_text != old_state:
        old_state = output_text
        print(outputs)
        print(output_text)

Noise Rejection Under Process...
tensor([[  3.8628, -12.1228]], device='cuda:0')
Eyes Open
tensor([[-3.7333,  6.0484]], device='cuda:0')
Eyes Closed
tensor([[  3.6051, -11.4946]], device='cuda:0')
Eyes Open


KeyboardInterrupt: 